In [ ]:
from chggen.common.sample_utils import CSP_Generator
from chggen.common.data_utils import mkdir
from chggen.common.sample_utils import get_inpaint_data_fromHost
from chggen.common.sample_utils import get_batch_inpaint_data_fromHost
from chggen.common.sample_utils import get_coarse_grain_framework, filter_nan_structure, compute_ewald_energy_single_structure

from types import SimpleNamespace
import numpy as np

from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core import Structure, Composition, Element, Lattice


import pandas as pd

import time
from datetime import datetime

In [ ]:
csp = CSP_Generator(chggen_path = "./files/cut_7_conv_3_epoch=27-val_loss=0.87.ckpt",
                    device='cuda:6')

In [ ]:
ld_kwargs = SimpleNamespace(
        n_step_each = 5,            # Corrector
        min_sigma = 0.01,
        num_noise_level = 200,
        signal_to_noise_ratio = 0.4,
        save_traj = False,
        disable_bar = False,
    )

In [ ]:
gen_kwargs = SimpleNamespace(
        num_gen = 3, # number of structures generated from the cubic lattice
        num_mutation = 2, # number of mutations during the relax-generation iteration
        num_cell = 1, # number of times to the formula
        ehull_cutoff = 0.06,
        )
                                           

In [ ]:
chemical_formula = 'MgSP2S5'
atomic_volume = 24
#  Generate seven different bravis lattices via diffusion
s_list_Bravis = csp.generate_structures_from_Bravis(comp_str= chemical_formula, atom_volume= atomic_volume,
                                                    gen_kwargs=gen_kwargs, ld_kwargs=ld_kwargs, )


s_list_Bravis = filter_nan_structure(s_list_Bravis)

In [ ]:
s_list_relax = []
for s in s_list_Bravis:
    atoms = AseAtomsAdaptor().get_atoms(s)

    result = csp.relaxer.relax(atoms= atoms,
                        fmax = 0.1,
                        steps = 2000,
                        relax_cell = True,
                        verbose = True,
                        # trajectory_path = None,
    )
    s_relax = result["final_structure"]
    s_list_relax.append(s_relax)

In [ ]:
# len(1.25)

In [ ]:
host_structure_list = []
num_intercalat_list = []

for s in s_list_relax:
    analyzer_asGen = SpacegroupAnalyzer(structure= s, symprec= 0.15, angle_tolerance= 15)
    symbol_asGen = analyzer_asGen.get_space_group_symbol()
    print("As generated spacegroup: ", symbol_asGen)

    s_frame, symbol_frame, num_species = get_coarse_grain_framework(s, species_to_remove= 'Li')
    s_frame = s_frame.get_primitive_structure()

    if symbol_frame in ['P1', 'P-1', 'Pm']: # or symbol_inpaint== 'P-1' or symbol_inpaint == 'Pm':
        continue

    host_structure_list.append(s_frame)
    num_intercalat_list.append(int(num_species))
    print("--"*10)

In [ ]:
len(host_structure_list)

In [ ]:
s_list_inpaint = csp.generate_from_host_structure(host_structure_list= host_structure_list * 3,
                                 num_intercalant_list= num_intercalat_list * 3,
                                 ld_kwargs=ld_kwargs, 
                                 species= 'Li')

In [ ]:
s_list_inpaint_conventional_unit = []

for s in s_list_inpaint:
    analyzer_inpaint = SpacegroupAnalyzer(structure= s, symprec= 0.2, angle_tolerance= 15)
    symbol_inpaint = analyzer_inpaint.get_space_group_symbol()
    

    

    if symbol_inpaint== 'P1' or symbol_inpaint== 'P-1':
        continue
    else:
        print(symbol_inpaint)
        s_inpaint_conventional_unit = analyzer_inpaint.get_conventional_standard_structure()
        s_list_inpaint_conventional_unit.append(s_inpaint_conventional_unit)

In [ ]:
E0_atom_list = []
for ii, s in enumerate(s_list_inpaint_conventional_unit):
    Ewald_per_atom = compute_ewald_energy_single_structure(s) / s.num_sites
    
    
    prediction = csp.chgnet.predict_structure(s)
    E0_atom = prediction['e']
    F_max = np.max(np.abs(prediction['f']))

    E0_atom_list.append(-E0_atom)
    # print("--"*10)
    # print(ii)
    # print(Ewald_per_atom, s.composition)

    # print("E0_per_atom: ", E0_atom, "F_max: ", F_max)

    # print("--"*10)

In [ ]:
# Calculate the threshold value
threshold = np.percentile(E0_atom_list, 50)

# Filter out the structures with E0_atom values above the threshold
filtered_structures = [structure for structure, e0_atom in zip(s_list_inpaint_conventional_unit, E0_atom_list) if e0_atom > threshold]


In [ ]:
np.sort(E0_atom_list)

In [ ]:
threshold

In [ ]:
filtered_structures

In [ ]:
ROOT ='files/inpaint_'+chemical_formula+'_LPS' 
mkdir(ROOT)

for ii, s in enumerate(filtered_structures):
    s.to(filename=ROOT +'/_'+str(ii)+'.cif')